# E[f²] Correction Study

Diagnose the source of E[f²] bias in the interaction SSNN and test correction approaches.

The arc-cosine kernel formula `relu_E_sigma_sigma(C)` overestimates E[f²] by 75–98% on
binomial genotype data. Two hypotheses:

1. **C_kl mismatch**: The 2×2 covariance matrix C_kl is computed from the Gaussian-latent
   Sigma, but the *true* covariance of binomial projections differs.
2. **Tail non-Gaussianity**: Even with the correct C_kl, binomial projections are non-Gaussian
   (discrete, bounded, skewed), so the arc-cosine formula is inexact.

This notebook decomposes the error to determine which factor dominates.

In [ ]:
import numpy as np
from scipy.stats import norm

from ssnn.activations import relu_E_sigma_sigma
from ssnn.gaussian_integrals import pairwise_covariance, projection_variance
from ssnn.edgeworth_integrals import edgeworth_E_sigma_sigma
from ssnn.cumulants import snp_cumulants, projection_cumulants_ld, decorrelation_matrix
from ssnn.simulation import (
    generate_maf_spectrum, generate_binomial_genotypes,
    generate_effect_sizes, SimulationScenario,
)
from ssnn.utils import generate_ld_matrix

## 1. Setup: Generate a test problem matching the interaction study

In [ ]:
p = 30
n = 200_000
rng = np.random.default_rng(42)

maf = generate_maf_spectrum(p, "common", rng)
Sigma = generate_ld_matrix(p, decay=0.5)
beta_star, sigma_eps = generate_effect_sizes(p, Sigma, 0.5, 0.2, rng)

X_binom = generate_binomial_genotypes(n, maf, Sigma, rng)
col_means = X_binom.mean(axis=0)
X_centered = X_binom - col_means

# Empirical covariance of binomial genotypes
Cov_emp = X_centered.T @ X_centered / n

# Also generate Gaussian data for comparison
X_gauss = rng.multivariate_normal(np.zeros(p), Sigma, size=n)
Cov_gauss_emp = X_gauss.T @ X_gauss / n

# Weight vectors (a few test directions)
w1 = rng.standard_normal(p) * 0.3
w2 = rng.standard_normal(p) * 0.3
w3 = beta_star / (np.linalg.norm(beta_star) + 1e-15) * 0.5

test_weights = [(w1, w2, "random w1, w2"),
                (w1, w1, "same direction (w1, w1)"),
                (w3, w1, "beta* direction vs random")]

## 2. Decompose E[ReLU(z_k) ReLU(z_l)] error

For each pair of weight vectors, compare:
- **MC truth (binomial)**: empirical average on binomial data
- **MC truth (Gaussian)**: empirical average on Gaussian data
- **Analytic with Sigma** (current): `relu_E_sigma_sigma(C_kl)` where `C_kl = w^T Sigma w`
- **Analytic with Cov_emp**: `relu_E_sigma_sigma(C_kl_emp)` where `C_kl_emp = w^T Cov_emp w`
- **Edgeworth-corrected**: `edgeworth_E_sigma_sigma(C_kl, kt3, kt4, ...)`

In [ ]:
Sigma_inv_sqrt = decorrelation_matrix(Sigma)

print(f"{'Pair':<28} {'MC binom':>10} {'MC gauss':>10} {'Analytic(Σ)':>12} "
      f"{'Analytic(Ĉ)':>12} {'Edgeworth':>10} "
      f"{'Err(Σ)%':>8} {'Err(Ĉ)%':>8} {'Err(EW)%':>9}")
print("-" * 130)

for wk, wl, label in test_weights:
    # MC truth on binomial data
    relu_k_binom = np.maximum(0.0, X_centered @ wk)
    relu_l_binom = np.maximum(0.0, X_centered @ wl)
    mc_binom = np.mean(relu_k_binom * relu_l_binom)

    # MC truth on Gaussian data
    relu_k_gauss = np.maximum(0.0, X_gauss @ wk)
    relu_l_gauss = np.maximum(0.0, X_gauss @ wl)
    mc_gauss = np.mean(relu_k_gauss * relu_l_gauss)

    # Analytic with Gaussian-latent Sigma
    C_sigma = pairwise_covariance(Sigma, wk, wl)
    analytic_sigma = relu_E_sigma_sigma(C_sigma)

    # Analytic with empirical covariance
    C_emp = np.array([
        [wk @ Cov_emp @ wk, wk @ Cov_emp @ wl],
        [wl @ Cov_emp @ wk, wl @ Cov_emp @ wl],
    ])
    analytic_emp = relu_E_sigma_sigma(C_emp)

    # Edgeworth correction (using Sigma-based C_kl)
    kt3_k, kt4_k = projection_cumulants_ld(wk, maf, Sigma, Sigma_inv_sqrt)
    kt3_l, kt4_l = projection_cumulants_ld(wl, maf, Sigma, Sigma_inv_sqrt)
    ew_val = edgeworth_E_sigma_sigma(C_sigma, kt3_k, kt4_k, kt3_l, kt4_l, "relu")

    err_sigma = (analytic_sigma - mc_binom) / (abs(mc_binom) + 1e-15) * 100
    err_emp = (analytic_emp - mc_binom) / (abs(mc_binom) + 1e-15) * 100
    err_ew = (ew_val - mc_binom) / (abs(mc_binom) + 1e-15) * 100

    print(f"{label:<28} {mc_binom:10.4f} {mc_gauss:10.4f} {analytic_sigma:12.4f} "
          f"{analytic_emp:12.4f} {ew_val:10.4f} "
          f"{err_sigma:7.1f}% {err_emp:7.1f}% {err_ew:8.1f}%")

## 3. Diagonal analysis: Sigma vs Cov_emp

Compare the diagonal entries (variances) and a few off-diagonal entries to understand
the structural difference between the Gaussian-latent Sigma and the empirical binomial covariance.

In [ ]:
print("Diagonal comparison: Sigma_jj vs Cov_emp_jj vs 2*p*(1-p)")
print(f"{'SNP':>4} {'MAF':>6} {'Sigma_jj':>10} {'Cov_emp_jj':>12} {'2p(1-p)':>10} {'Σ/Ĉ ratio':>10}")
print("-" * 60)
for j in range(min(p, 10)):
    binom_var = 2 * maf[j] * (1 - maf[j])
    ratio = Sigma[j, j] / (Cov_emp[j, j] + 1e-15)
    print(f"{j:4d} {maf[j]:6.3f} {Sigma[j,j]:10.4f} {Cov_emp[j,j]:12.4f} {binom_var:10.4f} {ratio:10.4f}")

print(f"\nOverall diagonal ratio: Sigma_diag / Cov_emp_diag = {np.mean(np.diag(Sigma) / np.diag(Cov_emp)):.4f}")
print(f"Frobenius norm ratio: ||Sigma|| / ||Cov_emp|| = {np.linalg.norm(Sigma) / np.linalg.norm(Cov_emp):.4f}")

## 4. Projection variance comparison

For each test weight vector, compare `w^T Sigma w` vs `w^T Cov_emp w` vs empirical `Var(w^T X)`.

In [ ]:
print(f"{'Weight':>25} {'w^T Σ w':>10} {'w^T Ĉ w':>10} {'Var_emp':>10} {'Σ/Var ratio':>12} {'Ĉ/Var ratio':>12}")
print("-" * 85)
for wk, _, label in test_weights:
    v_sigma = float(wk @ Sigma @ wk)
    v_emp_mat = float(wk @ Cov_emp @ wk)
    v_emp_direct = float(np.var(X_centered @ wk))
    print(f"{label:<25} {v_sigma:10.4f} {v_emp_mat:10.4f} {v_emp_direct:10.4f} "
          f"{v_sigma/v_emp_direct:12.4f} {v_emp_mat/v_emp_direct:12.4f}")

## 5. Full E[f²] comparison with network-scale weights

Use the Oracle NN weights and the Interaction NN weights from a simulation run
to compare the full E[f²] under different computation methods.

In [ ]:
from ssnn.optimizer import train
from ssnn.interaction_optimizer import train_interaction
from ssnn.simulation import train_oracle_nn, compute_summary_stats_from_genotypes

# Smaller dataset for training
n_train = 5000
rng2 = np.random.default_rng(123)
X_train_raw = generate_binomial_genotypes(n_train, maf, Sigma, rng2)
train_means = X_train_raw.mean(axis=0)
X_train = X_train_raw - train_means

w_star = rng2.standard_normal(p) * 0.3
gamma_coeff = 0.8
y_train = (X_train @ beta_star
           + gamma_coeff * np.maximum(0.0, X_train @ w_star)
           + rng2.normal(0, sigma_eps, n_train))

stats = compute_summary_stats_from_genotypes(X_train_raw, y_train, Sigma)
Cov_train = X_train.T @ X_train / n_train

# Train models
m = 5
gauss_res = train(Sigma, stats["Sigma_beta_hat"], stats["E_y2_hat"],
                  m=m, activation="relu", lr=0.01, max_iters=3000,
                  rng=np.random.default_rng(42))

int_res = train_interaction(Sigma, stats["Sigma_beta_hat"], stats["E_y2_hat"],
                            stats["Gamma_hat"], m=m, activation="relu",
                            lr=0.005, max_iters=3000,
                            a_init=gauss_res.a, W_init=gauss_res.W,
                            rng=np.random.default_rng(42))

oracle_a, oracle_W, _ = train_oracle_nn(X_train, y_train, m=m, activation="relu",
                                         lr=0.01, max_iters=5000,
                                         rng=np.random.default_rng(42))

print("Models trained.")

In [ ]:
def compute_E_f2_methods(a, W, X_data, Sigma, Cov_emp, maf, Sigma_inv_sqrt):
    """Compare E[f²] via different methods."""
    m = len(a)

    # MC truth
    hidden = np.maximum(0.0, X_data @ W.T)
    f_vals = hidden @ a
    mc_val = float(np.mean(f_vals ** 2))

    # Analytic with Sigma
    ef2_sigma = 0.0
    for k in range(m):
        for l in range(m):
            C_kl = pairwise_covariance(Sigma, W[k], W[l])
            ef2_sigma += a[k] * a[l] * relu_E_sigma_sigma(C_kl)

    # Analytic with Cov_emp
    ef2_cov = 0.0
    for k in range(m):
        for l in range(m):
            C_kl = np.array([
                [W[k] @ Cov_emp @ W[k], W[k] @ Cov_emp @ W[l]],
                [W[l] @ Cov_emp @ W[k], W[l] @ Cov_emp @ W[l]],
            ])
            ef2_cov += a[k] * a[l] * relu_E_sigma_sigma(C_kl)

    # Edgeworth with Sigma
    ef2_ew = 0.0
    for k in range(m):
        kt3_k, kt4_k = projection_cumulants_ld(W[k], maf, Sigma, Sigma_inv_sqrt)
        for l in range(m):
            kt3_l, kt4_l = projection_cumulants_ld(W[l], maf, Sigma, Sigma_inv_sqrt)
            C_kl = pairwise_covariance(Sigma, W[k], W[l])
            ef2_ew += a[k] * a[l] * edgeworth_E_sigma_sigma(
                C_kl, kt3_k, kt4_k, kt3_l, kt4_l, "relu")

    # Edgeworth with Cov_emp
    ef2_ew_cov = 0.0
    for k in range(m):
        kt3_k, kt4_k = projection_cumulants_ld(W[k], maf, Sigma, Sigma_inv_sqrt)
        for l in range(m):
            kt3_l, kt4_l = projection_cumulants_ld(W[l], maf, Sigma, Sigma_inv_sqrt)
            C_kl = np.array([
                [W[k] @ Cov_emp @ W[k], W[k] @ Cov_emp @ W[l]],
                [W[l] @ Cov_emp @ W[k], W[l] @ Cov_emp @ W[l]],
            ])
            ef2_ew_cov += a[k] * a[l] * edgeworth_E_sigma_sigma(
                C_kl, kt3_k, kt4_k, kt3_l, kt4_l, "relu")

    return {
        "mc": mc_val,
        "sigma": ef2_sigma,
        "cov_emp": ef2_cov,
        "edgeworth_sigma": ef2_ew,
        "edgeworth_cov": ef2_ew_cov,
    }


# Use the large binomial dataset for MC truth
X_eval = X_centered[:50_000]

models = [
    ("Gaussian NN", gauss_res.a, gauss_res.W),
    ("Interaction NN", int_res.a, int_res.W),
    ("Oracle NN", oracle_a, oracle_W),
]

print(f"{'Model':<18} {'MC truth':>10} {'Σ-analytic':>12} {'Ĉ-analytic':>12} "
      f"{'EW(Σ)':>10} {'EW(Ĉ)':>10} "
      f"{'Err(Σ)%':>9} {'Err(Ĉ)%':>9} {'Err(EW+Ĉ)%':>11}")
print("-" * 130)

for name, a, W in models:
    res = compute_E_f2_methods(a, W, X_eval, Sigma, Cov_emp, maf, Sigma_inv_sqrt)
    mc = res["mc"]
    err_s = (res["sigma"] - mc) / (abs(mc) + 1e-15) * 100
    err_c = (res["cov_emp"] - mc) / (abs(mc) + 1e-15) * 100
    err_ewc = (res["edgeworth_cov"] - mc) / (abs(mc) + 1e-15) * 100
    print(f"{name:<18} {mc:10.4f} {res['sigma']:12.4f} {res['cov_emp']:12.4f} "
          f"{res['edgeworth_sigma']:10.4f} {res['edgeworth_cov']:10.4f} "
          f"{err_s:8.1f}% {err_c:8.1f}% {err_ewc:10.1f}%")

## 6. Error decomposition summary

Attribute the total E[f²] error to:
- **C_kl mismatch**: error from using Sigma instead of Cov_emp
- **Residual non-Gaussianity**: remaining error after fixing C_kl

In [ ]:
print("\nError Decomposition")
print("=" * 80)
for name, a, W in models:
    res = compute_E_f2_methods(a, W, X_eval, Sigma, Cov_emp, maf, Sigma_inv_sqrt)
    mc = res["mc"]
    total_err = res["sigma"] - mc
    ckl_err = res["sigma"] - res["cov_emp"]
    residual_err = res["cov_emp"] - mc

    print(f"\n{name}:")
    print(f"  Total error (Σ - MC):        {total_err:+.4f} ({total_err/abs(mc)*100:+.1f}%)")
    print(f"  C_kl mismatch (Σ - Ĉ):      {ckl_err:+.4f} ({ckl_err/abs(mc)*100:+.1f}%)")
    print(f"  Residual non-Gauss (Ĉ - MC): {residual_err:+.4f} ({residual_err/abs(mc)*100:+.1f}%)")
    print(f"  C_kl accounts for:           {abs(ckl_err)/abs(total_err)*100:.1f}% of total error")

    # Also show Edgeworth+Cov_emp
    ew_cov_err = res["edgeworth_cov"] - mc
    print(f"  EW+Ĉ residual:              {ew_cov_err:+.4f} ({ew_cov_err/abs(mc)*100:+.1f}%)")

## 7. Impact on loss surface: does corrected E[f²] fix the Oracle penalty?

The key test: with corrected E[f²], does the Oracle's parameter region look better or
worse than the Interaction NN's?

In [ ]:
from ssnn.gaussian_integrals import stein_cross_moment
from ssnn.interaction_integrals import interaction_cross_moment

def compute_loss_variants(a, W, Sigma, Sigma_beta, E_y2, Gamma, Cov_emp, maf, Sigma_inv_sqrt):
    """Compute loss using different E[f²] methods."""
    m = len(a)

    # E[y*f] is the same for all variants
    E_yf = 0.0
    for k in range(m):
        E_yf += a[k] * (
            stein_cross_moment(Sigma, W[k], Sigma_beta, "relu")
            + interaction_cross_moment(Sigma, Gamma, W[k], "relu")
        )

    # E[f²] variants
    ef2_sigma = 0.0
    ef2_cov = 0.0
    for k in range(m):
        for l in range(m):
            C_sigma = pairwise_covariance(Sigma, W[k], W[l])
            ef2_sigma += a[k] * a[l] * relu_E_sigma_sigma(C_sigma)

            C_emp = np.array([
                [W[k] @ Cov_emp @ W[k], W[k] @ Cov_emp @ W[l]],
                [W[l] @ Cov_emp @ W[k], W[l] @ Cov_emp @ W[l]],
            ])
            ef2_cov += a[k] * a[l] * relu_E_sigma_sigma(C_emp)

    return {
        "E_yf": E_yf,
        "loss_sigma": E_y2 - 2 * E_yf + ef2_sigma,
        "loss_cov": E_y2 - 2 * E_yf + ef2_cov,
    }


from ssnn.utils import nn_prediction_r2

X_test_raw = generate_binomial_genotypes(5000, maf, Sigma, np.random.default_rng(999))
X_test = X_test_raw - train_means
y_test = (X_test @ beta_star
          + gamma_coeff * np.maximum(0.0, X_test @ w_star)
          + np.random.default_rng(999).normal(0, sigma_eps, 5000))

print(f"{'Model':<18} {'Loss(Σ)':>10} {'Loss(Ĉ)':>10} {'Test R²':>10} {'Test MSE':>10}")
print("-" * 65)
for name, a, W in models:
    lv = compute_loss_variants(a, W, Sigma, stats["Sigma_beta_hat"],
                               stats["E_y2_hat"], stats["Gamma_hat"],
                               Cov_train, maf, Sigma_inv_sqrt)
    r2 = nn_prediction_r2(X_test, y_test, a, W, "relu")
    pred = np.maximum(0.0, X_test @ W.T) @ a
    mse = float(np.mean((y_test - pred) ** 2))
    print(f"{name:<18} {lv['loss_sigma']:10.4f} {lv['loss_cov']:10.4f} {r2:10.4f} {mse:10.4f}")

print("\nKey question: Does Loss(Ĉ) rank the models correctly (Oracle < Interaction < Gaussian)?")

## 8. Conclusion

In [ ]:
# Summary of findings
print("SUMMARY")
print("=" * 60)
print("\nRun the cells above to populate results.")
print("\nKey metrics to check:")
print("  1. What % of E[f²] error is from C_kl mismatch?")
print("  2. Does Ĉ-analytic rank models correctly?")
print("  3. How much residual error after Ĉ correction?")